In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 47.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=3e25ab01141d9c8db4292ad51616ba8a02b9fe0e7d9531c2ec9b4fee6efabbc6
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# BB84 Quantum Key Distribution : simulation WITHOUT an attacker
#
# Alice and Bob establish a shared secret key over a quantum channel.
# With no eavesdropper, their sifted keys must be identical.

In [3]:
# Quantum Random Number Generator
# Each qubit is prepared in (|0> + |1>) / sqrt(2) by applying H to |0>.
# Measuring this state yields 0 or 1 with equal probability : true quantum randomness.

def quantum_random_bits(n):
    """Return n random bits from quantum measurement of n H|0> qubits."""
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)              # H|0> = (|0> + |1>) / sqrt(2)
    qc.measure(range(n), range(n))

    simulator = BasicSimulator()
    compiled  = transpile(qc, simulator)
    result    = simulator.run(compiled, shots=1).result()
    bitstring = list(result.get_counts().keys())[0]
    # Qiskit is little-endian: rightmost char = qubit 0, so reverse.
    return [int(b) for b in reversed(bitstring)]

In [4]:
# ALICE

# Alice generates N random bits (her intended key) and N random
# basis choices using the quantum RNG.
#
#   Basis 0 = rectilinear (+): bit 0 -> |0>,  bit 1 -> |1>
#   Basis 1 = diagonal    (x): bit 0 -> |+>,  bit 1 -> |->

N = 20   # number of qubits transmitted

alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)  # 0 = rectilinear, 1 = diagonal

print("=== ALICE ===")
print(f"Bits:  {alice_bits}")
print(f"Bases: {alice_bases}  (0=+, 1=x)")

=== ALICE ===
Bits:  [1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0]
Bases: [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]  (0=+, 1=x)


In [5]:
# QUANTUM CHANNEL : BOB

# Bob independently picks random measurement bases.
#
# The quantum channel is modelled as a single N-qubit circuit:
#
#   [Alice encodes] (barrier)  [Bob decodes]  measure
#
# Alice's encoding rules:
#   bit=0, basis=0: no gates  -> |0>
#   bit=1, basis=0: X         -> |1>
#   bit=0, basis=1: H         -> |+>
#   bit=1, basis=1: X then H  -> |->
#
# Bob's decoding rule:
#   basis=0: measure directly  (computational basis)
#   basis=1: apply H first     (rotates diagonal -> computational)

# --- BOB chooses measurement bases ---
bob_bases = quantum_random_bits(N)

print("=== BOB ===")
print(f"Bases: {bob_bases}  (0=+, 1=x)")
print()

# --- Build the combined Alice -> Bob circuit ---
qc = QuantumCircuit(N, N)

# Alice's encoding
for i in range(N):
    if alice_bits[i] == 1:
        qc.x(i)          # |0> -> |1>
    if alice_bases[i] == 1:
        qc.h(i)          # |0>->|+>,  |1>->|->

qc.barrier()             # visual separator: Alice side | Bob side

# Bob's decoding
for i in range(N):
    if bob_bases[i] == 1:
        qc.h(i)          # |+>->|0>,  |->->|1>

qc.measure(range(N), range(N))

# Show example circuit for qubit 0
print("Sample circuit for qubit 0 (Alice encodes | Bob decodes | measure):")
ex = QuantumCircuit(1, 1)
if alice_bits[0] == 1:  ex.x(0)
if alice_bases[0] == 1: ex.h(0)
ex.barrier()
if bob_bases[0] == 1:   ex.h(0)
ex.measure(0, 0)
print(ex.draw())

# --- Transmit: run the circuit once (one quantum transmission) ---
simulator = BasicSimulator()
compiled  = transpile(qc, simulator)
result    = simulator.run(compiled, shots=1).result()
bitstring = list(result.get_counts().keys())[0]
bob_results = [int(b) for b in reversed(bitstring)]

print(f"\nBob's measured bits: {bob_results}")

=== BOB ===
Bases: [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0]  (0=+, 1=x)

Sample circuit for qubit 0 (Alice encodes | Bob decodes | measure):
     ┌───┐ ░ ┌─┐
  q: ┤ X ├─░─┤M├
     └───┘ ░ └╥┘
c: 1/═════════╩═
              0 

Bob's measured bits: [1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0]


In [6]:
# CLASSICAL CHANNEL : Basis Comparison  (public)

# Alice and Bob publicly announce their basis choices (NOT bit values).
# They keep only positions where both used the same basis -> sifted key.

matching = [i for i in range(N) if alice_bases[i] == bob_bases[i]]

alice_sifted = [alice_bits[i]  for i in matching]
bob_sifted   = [bob_results[i] for i in matching]

print("=== BASIS COMPARISON ===")
header = f"{'Qubit':>5} | {'Alice':>6} | {'Bob':>6} | {'Keep?':>6}"
print(header)
print("-" * len(header))
for i in range(N):
    keep = "YES" if alice_bases[i] == bob_bases[i] else ""
    print(f"{i:>5} | {alice_bases[i]:>6} | {bob_bases[i]:>6} | {keep:>6}")

print(f"\nMatched {len(matching)} / {N} positions: {matching}")
print(f"\nAlice sifted key : {alice_sifted}")
print(f"Bob   sifted key : {bob_sifted}")

=== BASIS COMPARISON ===
Qubit |  Alice |    Bob |  Keep?
--------------------------------
    0 |      0 |      0 |    YES
    1 |      0 |      0 |    YES
    2 |      1 |      0 |       
    3 |      0 |      1 |       
    4 |      1 |      0 |       
    5 |      0 |      0 |    YES
    6 |      1 |      0 |       
    7 |      1 |      0 |       
    8 |      0 |      1 |       
    9 |      1 |      0 |       
   10 |      0 |      0 |    YES
   11 |      0 |      0 |    YES
   12 |      0 |      1 |       
   13 |      0 |      0 |    YES
   14 |      0 |      1 |       
   15 |      0 |      1 |       
   16 |      1 |      1 |    YES
   17 |      0 |      0 |    YES
   18 |      0 |      1 |       
   19 |      0 |      0 |    YES

Matched 9 / 20 positions: [0, 1, 5, 10, 11, 13, 16, 17, 19]

Alice sifted key : [1, 0, 1, 0, 1, 0, 1, 0, 0]
Bob   sifted key : [1, 0, 1, 0, 1, 0, 1, 0, 0]


In [7]:
# VERIFICATION : Key Agreement

# Without an attacker every matched bit should be identical.

errors     = sum(a != b for a, b in zip(alice_sifted, bob_sifted))
error_rate = errors / len(alice_sifted) if alice_sifted else 0.0

print("=== RESULT ===")
print(f"Sifted key length : {len(alice_sifted)} bits")
print(f"Errors            : {errors}")
print(f"Error rate        : {error_rate:.1%}")
print()
if error_rate == 0.0:
    print("[SUCCESS] Keys match perfectly.")
    print(f"Shared key: {alice_sifted}")
else:
    print("[FAILURE] Keys differ — unexpected without an attacker.")
    print(f"Alice : {alice_sifted}")
    print(f"Bob   : {bob_sifted}")

=== RESULT ===
Sifted key length : 9 bits
Errors            : 0
Error rate        : 0.0%

[SUCCESS] Keys match perfectly.
Shared key: [1, 0, 1, 0, 1, 0, 1, 0, 0]
